[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Missing Values &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]
X_train, y_train = train[FEATURES], train["tmin"]
X_valid, y_valid = valid[FEATURES], valid["tmin"]
X_test, y_test = test[FEATURES], test["tmin"]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' helpers and median, rebuilt here from the same made-up stations so that each answer
runs on its own.

In [2]:
def lightgbm_validation(train_X, valid_X, **settings):
    model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.1, random_state=0, verbose=-1, **settings)
    model.fit(train_X, y_train, eval_set=[(valid_X, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])
    return model, model.best_iteration_, round(float(model.best_score_["valid_0"]["l2"]), 3)


def xgboost_validation(train_X, valid_X, **settings):
    model = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.1, early_stopping_rounds=100, random_state=0,
                             **settings)
    model.fit(train_X, y_train, eval_set=[(valid_X, y_valid)], verbose=False)
    return model, model.best_iteration + 1, round(mean_squared_error(y_valid, model.predict(valid_X)), 3)


median = X_train["snow_cm"].median()

**1.** Print the share of the test rows with a gap in `snow_cm`, and the number of test stations with no snow sensor.

In [3]:
no_sensor = test.groupby("station")["snow_cm"].apply(lambda readings: readings.isna().all())
print("share of test rows with a gap:", round(test["snow_cm"].isna().mean(), 3))
print("test stations with no sensor:", no_sensor.sum(), "of", len(no_sensor))

share of test rows with a gap: 0.3
test stations with no sensor: 15 of 50


Three tenths of the test rows have a gap, from 15 of the 50 test stations.

**2.** Fit LightGBM's stump on `snow_cm` alone, as in the worked examples, and print the mean answer on each side of its split, with the gaps counted on the side its `missing_direction` names.

In [4]:
stump = lgb.LGBMRegressor(n_estimators=1, num_leaves=2, learning_rate=1.0, verbose=-1)
stump.fit(X_train[["snow_cm"]], y_train)
root = stump.booster_.trees_to_dataframe().iloc[0]

snow, gap = X_train["snow_cm"], X_train["snow_cm"].isna()
left = (snow <= root["threshold"]) | (gap if root["missing_direction"] == "left" else False)
print("split at", round(root["threshold"], 3), "| gaps go", root["missing_direction"])
print("mean answer on the left:", round(y_train[left].mean(), 3), "over", left.sum(), "rows")
print("mean answer on the right:", round(y_train[~left].mean(), 3), "over", (~left).sum(), "rows")

split at 10.5 | gaps go left
mean answer on the left: 0.909 over 10735 rows
mean answer on the right: -7.642 over 4265 rows


LightGBM's stump splits at 10.5 cm and sends the gaps left, so the left side holds 10,735 rows, the days below the threshold and the gaps together, with a mean of 0.909 °C, and the right side the 4,265 snowy days at −7.642 °C.

**3.** Add a column `has_sensor`, 1 where the snow was measured and 0 where it was not, fill the gaps with the training median, and print both libraries' trees and validation losses with the worked examples' helpers.

In [5]:
flagged = lambda X: X.assign(has_sensor=X["snow_cm"].notna().astype(int)).fillna({"snow_cm": median})
_, trees, loss = lightgbm_validation(flagged(X_train), flagged(X_valid))
print(f"LightGBM: {trees} trees, validation loss {loss}")
_, trees, loss = xgboost_validation(flagged(X_train), flagged(X_valid))
print(f"XGBoost:  {trees} trees, validation loss {loss}")

LightGBM: 148 trees, validation loss 2.559
XGBoost:  237 trees, validation loss 2.525


With the flag and a median fill, LightGBM stops after 148 trees at 2.559 and XGBoost after 237 at 2.525, within 0.02 of the gaps left in, 2.546 and 2.526, in one run each.

**4.** Make a copy of the training rows with a gap in `cloud_cover` on 10% of them, chosen with `np.random.default_rng(0)`, train LightGBM on the copy, and print the missing types of its splits on `cloud_cover`.

In [6]:
cloudy = X_train.copy()
rows = np.random.default_rng(0).choice(len(cloudy), len(cloudy) // 10, replace=False)
cloudy.iloc[rows, cloudy.columns.get_loc("cloud_cover")] = np.nan

model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=0, verbose=-1).fit(cloudy, y_train)
splits = model.booster_.trees_to_dataframe()
print("gaps in cloud_cover:", cloudy["cloud_cover"].isna().sum())
print("missing types on cloud_cover:", splits[splits["split_feature"] == "cloud_cover"]["missing_type"].unique().tolist())

gaps in cloud_cover: 1500
missing types on cloud_cover: ['NaN']


With 1,500 gaps in `cloud_cover`, every LightGBM split on it has the missing type `NaN`, so a gap follows each split's `missing_direction` instead of being read as 0.

**5.** Make a copy of the validation stations with a gap in `snow_cm` on every row, and print each library's validation loss on it beside its loss on the real readings.

In [7]:
no_snow = X_valid.assign(snow_cm=np.nan)
lightgbm_model, _, _ = lightgbm_validation(X_train, X_valid)
xgboost_model, _, _ = xgboost_validation(X_train, X_valid)
for name, model in (("LightGBM", lightgbm_model), ("XGBoost", xgboost_model)):
    print(f"{name}: {mean_squared_error(y_valid, model.predict(X_valid)):.3f} with the readings, "
          f"{mean_squared_error(y_valid, model.predict(no_snow)):.3f} with every reading a gap")

LightGBM: 2.546 with the readings, 2.708 with every reading a gap
XGBoost: 2.526 with the readings, 2.737 with every reading a gap


With every snow reading a gap, the validation loss rises from 2.546 to 2.708 for LightGBM and from 2.526 to 2.737 for XGBoost. Both models had gaps in `snow_cm` in training, so each gap follows a split's default direction, and the loss rises by what the readings at the 34 validation stations with a sensor were worth.

---

&#8592; **Back to:** [Missing Values](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/08-missing-values.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
